# Best Local LLMs in Practice (2026 Edition)

This notebook demonstrates how to use the top-performing open-source language models that can run locally with <64GB RAM. We'll focus on the two families this course uses: Gemma 4 (the course default) and Qwen 3.8.

## Prerequisites

- Install Ollama: https://ollama.ai/
- At least 16GB RAM (32GB+ recommended)
- GPU with 8GB+ VRAM (optional but recommended)

## 1. Gemma 4 — The 2026 Course Default

Gemma 4 (Google DeepMind, April 2026) is the **course default** for this session. Pick the size that fits your hardware:
- **`gemma4:e4b`** (9.6 GB) — what bare `gemma4` pulls; CPU-friendly, no GPU required
- **`gemma4:e2b`** (7.2 GB) — smallest variant for low-RAM machines
- **`gemma4:12b`** (≈8 GB download) — targets 16 GB VRAM / unified-memory laptops
- **`gemma4:26b`** (MoE, 19 GB) / **`gemma4:31b`** (20 GB) — for 24 GB GPUs
- **Multimodal** — image input on all sizes; audio on E2B/E4B/12B only (via Hugging Face Transformers, not exposed through Ollama)
- **Apache 2.0** — no MAU caps, no EU restrictions


In [1]:
import ollama
from IPython.display import Markdown, display

def chat_with_gemma4(prompt, model="gemma4"):
    """Chat with Gemma 4 via Ollama — course default model."""
    try:
        response = ollama.chat(
            model=model,
            messages=[{'role': 'user', 'content': prompt}]
        )
        return response['message']['content']
    except Exception as e:
        return f"Error: {e}. Run 'ollama pull {model}' first."

# `gemma4` = gemma4:e4b (CPU-friendly). On a 16 GB laptop try model="gemma4:12b";
# on a 24 GB GPU, model="gemma4:26b" or "gemma4:31b".
response = chat_with_gemma4(
    "Explain in 2 sentences why running LLMs locally matters for privacy."
)
display(Markdown(response))


Cloud-based LLMs require transmitting all input data over the internet, creating potential vulnerabilities where sensitive personal information can be intercepted, logged, or analyzed by external servers. Running models locally ensures that your data—including your queries and the resulting outputs—remains entirely on your device, maintaining maximum confidentiality and control.

## 2. Qwen 3.8 27B — Coding + Agentic Tasks

Qwen 3.8 27B (Alibaba, Aug 2026, Apache 2.0, 18 GB) is the newer Qwen release (after Qwen 3.6) and the course's tool-calling alternative to Gemma 4:
- **27B parameters** — `qwen3.8:27b` is an 18 GB download; plan for a 24 GB GPU (or slow CPU offload)
- **Tool calling + coding** — tuned for coding and long-horizon agentic tasks
- **Thinking mode** — on by default, can be disabled per request
- **256K context window**, text + image input


In [2]:
def chat_with_qwen(prompt, model="qwen3.8:27b-mlx"):
    """Chat with Qwen 3.8 27B via Ollama (MLX build, Apple Silicon) — alternative for agentic/coding tasks."""
    try:
        response = ollama.chat(
            model=model,
            messages=[{'role': 'user', 'content': prompt}]
        )
        return response['message']['content']
    except Exception as e:
        return f"Error: {e}. Run 'ollama pull {model}' first."

response = chat_with_qwen(
    "Write a Python function that retries a failing HTTP request with exponential backoff."
)
display(Markdown(response))


```python
"""
HTTP request retry with exponential backoff and jitter.

Usage:
    response = request_with_retry("GET", "https://api.example.com/data")
"""

import time
import random
import logging
from typing import Optional, Sequence

import requests
from requests.exceptions import ConnectionError, Timeout, RequestException

logger = logging.getLogger(__name__)


def request_with_retry(
    method: str,
    url: str,
    *,
    max_retries: int = 5,
    base_delay: float = 1.0,
    max_delay: float = 60.0,
    jitter: float = 0.25,
    retryable_status_codes: Sequence[int] = (429, 500, 502, 503, 504),
    timeout: float = 30.0,
    **request_kwargs,
) -> requests.Response:
    """
    Perform an HTTP request with exponential backoff retry logic.

    Args:
        method:  HTTP method (GET, POST, PUT, DELETE, ...).
        url:     Target URL.
        max_retries:   Maximum number of retry attempts (not counting the first).
        base_delay:    Initial delay in seconds before the first retry.
        max_delay:     Upper bound on the delay in seconds.
        jitter:      Randomness factor (0.0–1.0) to prevent thundering-herd.
        retryable_status_codes: HTTP status codes that should trigger a retry.
        timeout:     Per-request socket timeout in seconds.
        **request_kwargs: Extra keyword arguments forwarded to requests.request().

    Returns:
        A requests.Response object (the last attempt's response on success).

    Raises:
        RequestException: If all retries are exhausted or a non-retryable error occurs.
    """
    last_exception: Optional[Exception] = None

    for attempt in range(max_retries + 1):
        try:
            response = requests.request(
                method,
                url,
                timeout=timeout,
                **request_kwargs,
            )

            # ---- Retry on transient HTTP status codes ----
            if response.status_code in retryable_status_codes:
                if attempt == max_retries:
                    response.raise_for_status()  # let the caller handle it
                    break

                delay = _compute_delay(attempt, base_delay, max_delay, jitter)
                logger.warning(
                    "Attempt %d/%d got HTTP %d for %s %s — retrying in %.2fs",
                    attempt + 1,
                    max_retries + 1,
                    response.status_code,
                    method,
                    url,
                    delay,
                )
                time.sleep(delay)
                continue

            # ---- Success (or non-retryable 4xx, etc.) ----
            return response

        # ---- Retry on network-level errors ----
        except (ConnectionError, Timeout) as exc:
            last_exception = exc
            if attempt == max_retries:
                logger.error(
                    "Attempt %d/%d failed for %s %s: %s (giving up)",
                    attempt + 1,
                    max_retries + 1,
                    method,
                    url,
                    exc,
                )
                raise

            delay = _compute_delay(attempt, base_delay, max_delay, jitter)
            logger.warning(
                "Attempt %d/%d failed for %s %s (%s) — retrying in %.2fs",
                attempt + 1,
                max_retries + 1,
                method,
                url,
                type(exc).__name__,
                delay,
            )
            time.sleep(delay)

        # ---- Unexpected errors: propagate immediately ----
        except RequestException as exc:
            logger.error("Non-retryable error on %s %s: %s", method, url, exc)
            raise

    # Should be unreachable, but keeps type-happeners happy
    raise last_exception  # type: ignore[misc]


def _compute_delay(
    attempt: int,
    base_delay: float,
    max_delay: float,
    jitter: float,
) -> float:
    """
    Compute the backoff delay for a given attempt number.

    Formula (Full Jitter, AWS-recommended):
        delay = random_between(0, min(max_delay, base_delay * 2 ** attempt))

    When jitter == 0 the formula degenerates to a pure exponential schedule.
    """
    exp_delay = min(max_delay, base_delay * (2 ** attempt))

    if jitter > 0:
        # Apply jitter: scale within [exp_delay*(1-jitter), exp_delay*(1+jitter)]
        low = exp_delay * (1.0 - jitter)
        high = exp_delay * (1.0 + jitter)
        return random.uniform(low, high)

    return exp_delay


# ---------------------------------------------------------------------------
# Convenience wrappers
# ---------------------------------------------------------------------------

def get_with_retry(url: str, **kwargs) -> requests.Response:
    return request_with_retry("GET", url, **kwargs)


def post_with_retry(url: str, **kwargs) -> requests.Response:
    return request_with_retry("POST", url, **kwargs)


# ---------------------------------------------------------------------------
# Usage examples
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

    # Basic GET with default backoff
    resp = request_with_retry("GET", "https://httpbin.org/get")
    print(resp.json())

    # POST with custom backoff: start at 2 s, cap at 30 s, 10 retries
    resp = request_with_retry(
        "POST",
        "https://httpbin.org/post",
        json={"key": "value"},
        max_retries=10,
        base_delay=2.0,
        max_delay=30.0,
        jitter=0.3,
    )
    print(resp.status_code)
```

## How it works

### Backoff schedule

| Attempt | Pure exponential (base=1s) | With 25 % jitter (range) |
|---------|----------------------------|--------------------------|
| 1 | 1 s | 0.75 – 1.25 s |
| 2 | 2 s | 1.5 – 2.5 s |
| 3 | 4 s | 3 – 5 s |
| 4 | 8 s | 6 – 10 s |
| 5 | 16 s (capped at `max_delay`) | 12 – 20 s |

The formula is `min(max_delay, base_delay × 2^attempt)`, then scaled by a random factor in `[1−jitter, 1+jitter]` to avoid the **thundering-herd** problem when many clients retry simultaneously.

### What triggers a retry

| Cause | Retried? |
|---|---|
| HTTP 429 / 500 / 502 / 503 / 504 | ✅ (configurable via `retryable_status_codes`) |
| `ConnectionError`, `Timeout` | ✅ |
| HTTP 400 / 401 / 403 / 404 | ❌ propagated immediately |
| Any other `RequestException` | ❌ propagated immediately |

### Key design choices

- **Jitter** prevents all concurrent clients from hammering the server at the same instant after a failure.
- **`max_delay` cap** prevents the exponential curve from growing unbounded (e.g., 2¹⁰ = 1024 s).
- **Non-retryable errors are not swallowed** — a 404 or a malformed URL fails fast.
- The function is **synchronous**; for `asyncio` / `httpx` use the same pattern with `await asyncio.sleep(...)` and `httpx.AsyncClient`.

### Optional: as a decorator

```python
import functools

def retry_requests(max_retries=5, **backoff_kwargs):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(max_retries + 1):
                try:
                    return func(*args, **kwargs)
                except (ConnectionError, Timeout) as exc:
                    if attempt == max_retries:
                        raise
                    delay = _compute_delay(attempt, **backoff_kwargs)
                    time.sleep(delay)
        return wrapper
    return decorator

# Usage
@retry_requests(max_retries=3, base_delay=1.0, max_delay=30.0, jitter=0.3)
def fetch_data():
    return requests.get("https://api.example.com/data", timeout=10)
```

For production systems that need to respect `Retry-After` headers, parse that header in the 429/503 branch and use `max(delay, float(retry_after))` instead of the computed delay.

## 3. Model Comparison - Same Task, Different Strengths

Let's give the same complex task to different models to see their unique approaches:

In [3]:
comparison_prompt = """
Design a distributed system for real-time fraud detection that can handle:
- 100,000 transactions per second
- Sub-100ms latency requirement
- 99.99% availability
- Global deployment

Provide:
1. High-level architecture diagram (in text/ASCII)
2. Technology stack recommendations
3. Scaling strategy
4. Key challenges and solutions

Keep it practical and implementable.
"""

print("=== GEMMA 4 RESPONSE ===")
gemma_response = chat_with_gemma4(comparison_prompt)
display(Markdown(gemma_response))

print("\n\n=== QWEN 3.8 RESPONSE ===")
qwen_response = chat_with_qwen(comparison_prompt)
display(Markdown(qwen_response))

=== GEMMA 4 RESPONSE ===


This design focuses on an event-driven, stateless, and highly parallelized architecture to meet the demanding requirements of 100,000 TPS and sub-100ms latency globally.

---

## 1. High-Level Architecture Diagram (Text/ASCII)

We utilize a **Multi-Region Active-Active** setup to ensure global availability and low latency regardless of the transaction source.

```mermaid
graph LR
    subgraph Global Edge Layer (CDNs/Gateways)
        A[Transaction Ingress (Client/API Gateways)] --> B{Load Balancing/Global Router};
    end

    subgraph Region A (e.g., US-East)
        B --> C1[Ingestion Topic (Kafka Cluster)];
        C1 --> D1[Stream Processor (Flink/Beam)];
        D1 --> E1{Feature Store Lookup (Redis/Aerospike)};
        E1 --> F1[Model Inference Engine (Microservice)];
        F1 --> G1[Decision Service (Rule Engine/ML)];
        G1 --> H1[Action Topic (Kafka)];
        H1 --> I1[Feedback Loop/Sink];
        I1 --> J1[Real-time Storage/Monitoring];
    end

    subgraph Region B (e.g., EU-West)
        B --> C2[Ingestion Topic (Kafka Cluster)];
        C2 --> D2[Stream Processor (Flink/Beam)];
        D2 --> E2{Feature Store Lookup (Redis/Aerospike)};
        E2 --> F2[Model Inference Engine (Microservice)];
        F2 --> G2[Decision Service (Rule Engine/ML)];
        G2 --> H2[Action Topic (Kafka)];
        H2 --> I2[Feedback Loop/Sink];
        I2 --> J2[Real-time Storage/Monitoring];
    end

    subgraph Data Layer (Asynchronous)
        J1 --> K[Data Lake (S3/GCS)];
        J2 --> K;
        I1 --> L[ML Training Pipeline (Spark/Beam)];
    end

    style B fill:#f9f,stroke:#333,stroke-width:2px
    style C1 fill:#ccf,stroke:#333
    style C2 fill:#ccf,stroke:#333
    style F1 fill:#ffc,stroke:#333
    style G1 fill:#ffc,stroke:#333
```

### Workflow Explanation:

1.  **Ingress:** Transactions hit the nearest global endpoint (CDN/API Gateway).
2.  **Ingestion:** The Gateway immediately publishes the raw transaction event to the regional Kafka cluster. This acts as a non-blocking queue, decoupling the ingestion rate from the processing rate.
3.  **Stream Processing:** The Stream Processor (e.g., Flink) consumes the event. Its primary tasks are:
    *   *Feature Extraction:* Calculating rolling aggregates (e.g., "number of transactions in the last 5 minutes for this user").
    *   *Enrichment:* Joining the transaction data with necessary contextual data.
4.  **Feature/State Lookup:** The processor queries the ultra-low-latency **Feature Store** (Redis/Aerospike) to retrieve up-to-date user/account state (e.g., last known location, spending velocity).
5.  **Inference & Decision:** The request passes through the Model Inference Engine (calling pre-trained models) and then the Rule Engine (applying business rules, e.g., "If ML Score > 0.8 AND amount > $500, block").
6.  **Action:** The Decision Service determines the final action (ALLOW, REVIEW, BLOCK) and publishes the result to an **Action Topic**.
7.  **Feedback & Learning:** The result is consumed by the backend systems (e.g., payment APIs) and simultaneously logged to the Data Lake and used to update the Feature Store state.

---

## 2. Technology Stack Recommendations

| Component | Technology Recommendation | Justification |
| :--- | :--- | :--- |
| **Ingestion/Messaging** | **Apache Kafka (MSK/Confluent)** | Highly scalable, durable, and provides guaranteed ordering required for financial transactions. Essential for handling burst loads and decoupling services. |
| **Stream Processing** | **Apache Flink** (or ksqlDB/Beam) | Best-in-class for stateful stream processing, event time handling, and complex windowed aggregation needed for real-time feature calculation. |
| **Feature/State Store** | **Redis Enterprise / Aerospike** | Critical for sub-millisecond lookups. Stores volatile, time-sensitive features (e.g., rolling counts, velocity metrics) that must be immediately available for scoring. |
| **Model Serving** | **Triton Inference Server (NVIDIA) / FastAPI** | Optimized for high-throughput, low-latency GPU/CPU model serving (Python/TensorFlow/PyTorch). Allows horizontal scaling of prediction services. |
| **Rule Engine** | **Drools / Custom Microservice** | Enables separation of complex business logic (rules) from the ML model, allowing fraud analysts to update rules without deploying new code. |
| **API Gateway/Edge** | **Cloudflare / AWS Global Accelerator** | Provides geo-routing, DDoS protection, and minimizes network latency by directing clients to the nearest operational region. |
| **Data Storage (Analytics)** | **Cloud Data Lake (S3/GCS)** | Cost-effective storage for all raw and processed historical data, used for model retraining and deep investigation. |
| **Deployment/Coordination**| **Kubernetes (K8s) & Service Mesh (Istio)** | Provides standardized orchestration, service discovery, health checks, and traffic routing across microservices in a multi-region setup. |

---

## 3. Scaling Strategy

The architecture employs three primary scaling dimensions: **Horizontal, State, and Geopolitical.**

### A. Horizontal Scaling (TPS Handling)
*   **Stateless Scaling:** The Model Inference Engine and Rule Engine must be kept as stateless microservices running on K8s. Scaling is achieved simply by increasing the replica count (Horizontal Pod Autoscaler).
*   **Kafka Partitioning:** The Kafka topics must be heavily partitioned (e.g., 1000+ partitions). The transaction ID (e.g., `User_ID`) must be used as the Kafka key to ensure all related events for a single user are processed sequentially by the same processor instance, maintaining correct state.
*   **Consumer Groups:** Scaling the Stream Processors (Flink/Beam) by adding worker nodes/pods ensures that the processing workload is automatically distributed across the available consumer group partitions.

### B. State Scaling (Feature Store)
*   The Feature Store (Redis/Aerospike) must be clustered and sharded. Data sharding should be based on the **User ID** or **Account ID**. This prevents a single large shard from becoming a bottleneck and distributes the read/write load evenly.
*   Since transaction feature calculation (e.g., `count_last_5_min`) is inherently stateful, the stream processor (Flink) manages this state internally and checkpoints it periodically to a highly available, distributed storage layer (like RocksDB within Flink, backed by Kafka).

### C. Geopolitical Scaling (Global Availability)
*   **Active-Active-Active:** Deploy the entire stack (Gateway, Kafka, Flink, Inference Engine) across at least three distinct global regions (e.g., US, EU, Asia).
*   **Global Load Balancing:** Use Global DNS/CDN routing to direct the user traffic to the nearest functional regional endpoint.
*   **Data Synchronization:** The **Feature Store** requires careful replication. Instead of replicating *all* user data, only highly volatile, localized state (like recent spending history) needs near-real-time replication between regions, while core static data (like country codes) can be globally cached.

---

## 4. Key Challenges and Solutions

| Challenge | Impact on System | Mitigation/Solution |
| :--- | :--- | :--- |
| **State Consistency** | If multiple instances process the same user's transaction out of order, feature calculations will be wrong, leading to missed fraud detections. | **Solution:** Use Kafka keying by `User_ID` to guarantee processing order. Use Flink's built-in **checkpointing** and **Watermarking** to handle event time ordering and recover state accurately upon failure. |
| **Handling Burst Traffic** | A sudden surge (e.g., flash sale) could overwhelm the feature store lookups or the inference engine. | **Solution:** **Backpressure and Queueing.** Kafka acts as a massive buffer, absorbing the burst. The Stream Processor (Flink) must be configured with proper backpressure mechanisms to slow down if the downstream component (e.g., Redis) is overloaded. Implement **Rate Limiting** at the API Gateway. |
| **Low Latency Feature Store Lookups** | Database calls are the most likely point of failure for the sub-100ms SLA. | **Solution:** Use **In-Memory Data Stores** (Redis/Aerospike). Structure the data queries to be simple key lookups (`GET user:X:count`) rather than complex joins or queries. Pre-calculate and store rolling aggregates. |
| **Model Staleness/Drift** | Fraud patterns change constantly. Models trained on old data will fail. | **Solution:** Implement a continuous ML pipeline (MLOps). The **Feedback Loop** must capture transaction outcomes (True/False fraud) and feed them into the training pipeline (e.g., triggered weekly or when performance drops) to retrain and redeploy models automatically. |
| **Global Data Sovereignty** | Different regions require data to stay within national boundaries (GDPR, CCPA). | **Solution:** Implement **Regional Data Isolation.** Kafka topics and databases must be physically segregated by region. Only anonymized, aggregated, or non-PII feature vectors should be allowed to sync globally for model improvement. |



=== QWEN 3.8 RESPONSE ===


# Distributed Real-Time Fraud Detection System

## 1. High-Level Architecture

```
                        ┌─────────────────────────────────────────────────────────┐
                        │                    GLOBAL CDN / EDGE                    │
                        │          (CloudFlare / AWS CloudFront / GCP CDN)        │
                        │   TLS termination  ·  geo-routing  ·  rate-limiting    │
                        └──────────────────────────┬──────────────────────────────┘
                                                   │
              ┌────────────────────────────────────┼────────────────────────────────────┐
              │                                    │                                    │
     ┌────────▼────────┐              ┌────────────▼────────┐              ┌───────────▼──────────┐
     │  REGION: us-east│              │   REGION: eu-west    │              │  REGION: ap-southeast │
     │  (active)       │              │  (active)            │              │  (active)             │
     │                 │◄─── peer ──►│  (active)            │◄── peer ───►│  (active)            │
     └────────┬────────┘              └────────────┬────────┘              └───────────┬──────────┘
              │                                    │                                    │
              ▼                                    ▼                                    ▼
┌─────────────────────────────────────────────────────────────────────────────────────────────────────┐
│                                                                                                     │
│  ┌──────────────────────────────────────────────────────────────────────────────────────────────┐    │
│  │                        API GATEWAY LAYER (per region)                                        │    │
│  │   Kong / Envoy  ·  JWT auth  ·  request dedup  ·  per-tenant rate-limiting  ·  circuit break │    │
│  └──────────────────────────────┬───────────────────────────────────────────────────────────────┘    │
│                                 │  (JSON / Protobuf, <3 ms)                                        │
│  ┌──────────────────────────────▼───────────────────────────────────────────────────────────────┐    │
│  │                    EVENT INGESTION (per region)                                                │    │
│  │                                                                                               │    │
│  │   ┌─────────────┐     ┌──────────────────┐     ┌──────────────────┐                          │    │
│  │   │  Kafka /    │     │  Kafka /         │     │  Kafka /         │   (partitioned by        │    │
│  │   │  Pulsar     │────►│  Pulsar          │────►│  Pulsar          │    txn_id / cust_id,     │    │
│  │   │  topic:     │     │  topic:          │     │  topic:          │    128 partitions each,  │    │
│  │   │  raw_txns   │     │  enriched_txns   │     │  decisions       │    min 3-broker per      │    │
│  │   │  (100K TPS) │     │  (100K TPS)      │     │  (100K TPS)      │    region, 3-replica)    │    │
│  │   └─────────────┘     └────────┬─────────┘     └────────┬─────────┘                          │    │
│  │                                │                         │                                      │    │
│  └────────────────────────────────│─────────────────────────│──────────────────────────────────────┘    │
│                                   │                         │                                          │
│  ┌────────────────────────────────▼─────────────────────────▼──────────────────────────────────────┐    │
│  │                  REAL-TIME PROCESSING LAYER (per region, Flink job)                              │    │
│  │                                                                                                  │    │
│  │   ┌─────────────────────────────────────────────────────────────────────────────────────────┐    │    │
│  │   │  STAGE 1 – FEATURE ENRICHMENT  (async side, ~15 ms budget)                            │    │    │
│  │   │                                                                                       │    │    │
│  │   │   ┌──────────────┐  ┌──────────────┐  ┌──────────────┐  ┌─────────────────────┐      │    │    │
│  │   │  │ Redis Cluster │  │ ScyllaDB /   │  │ Kafka state  │  │  Graph service       │      │    │    │
│  │   │  │  (hot feats:  │  │ Cassandra    │  │  store in    │  │  (Neo4j / Kùzu:      │      │    │    │
│  │   │  │  velocity,    │  │ (cold feats: │  │  Flink state  │  │  device graph,       │      │    │    │
│  │   │  │  device id,   │  │  90-day hist,│  │  backend      │  │  peer-network,       │      │    │    │
│  │   │  │  geo, IP,     │  │  account age,│  │  (RocksDB)    │  │  money-mule chains)  │      │    │    │
│  │   │  │  merchant cat)│  │  KYC flag)   │  │               │  │                     │      │    │    │
│  │   │  └──────┬───────┘  └──────┬───────┘  └──────┬───────┘  └────────┬────────────┘      │    │    │
│  │   │         └──────────────────┴───────────────────┴──────────────────┘                   │    │    │
│  │   │                          (all in-memory / <5 ms P99)                                  │    │    │
│  │   └─────────────────────────────────────────────────────────────────────────────────────────┘    │    │
│  │                                                                                                  │    │    │
│  │   ┌─────────────────────────────────────────────────────────────────────────────────────────┐    │    │
│  │   │  STAGE 2 – RULE ENGINE  (synchronous, ~5 ms budget)                                    │    │    │
│  │   │                                                                                       │    │    │
│  │   │   Drools / custom Go ruleset (compiled, in-memory)                                   │    │    │
│  │   │   · Velocity rules  · Blacklist match · Geo-velocity · Channel anomaly               │    │    │
│  │   │   · Rules hot-reloaded via Config Server (no redeploy)                                │    │    │
│  │   │   · If ANY rule fires → immediate HOLD / REJECT (short-circuit, skip ML)              │    │    │
│  │   └─────────────────────────────────────────────────────────────────────────────────────────┘    │    │
│  │                                                                                                  │    │    │
│  │   ┌─────────────────────────────────────────────────────────────────────────────────────────┐    │    │
│  │   │  STAGE 3 – ML SCORE  (synchronous, ~25 ms budget, runs only if rules pass)             │    │    │
│  │   │                                                                                       │    │    │
│  │   │   ┌──────────────────────────────────────────────────────────────┐                     │    │    │
│  │   │   │  Model Serving (per region, GPU node or CPU-optimized)      │                     │    │    │
│  │   │   │  · GBDT (LightGBM / XGBoost) via ONNX Runtime / TensorRT   │                     │    │    │
│  │   │   │  · 2-layer MLP for sequence anomalies (PyTorch→ONNX)       │                     │    │    │
│  │   │   │  · 200 features, batch size 1, P99 < 15 ms on CPU         │                     │    │    │
│  │   │   │  · 2 active replicas, 1 warm standby, canary model slots    │                     │    │    │
│  │   │   └──────────────────────────────────────────────────────────────┘                     │    │    │
│  │   └─────────────────────────────────────────────────────────────────────────────────────────┘    │    │
│  │                                                                                                  │    │    │
│  │   ┌─────────────────────────────────────────────────────────────────────────────────────────┐    │    │
│  │   │  STAGE 4 – DECISION & FEEDBACK  (~5 ms)                                                 │    │    │
│  │   │  · Combine rule score + ML score + risk tier → APPROVE / HOLD / REJECT / REVIEW         │    │    │
│  │   │  · Write decision to Kafka `decisions` topic (async)                                    │    │    │
│  │   │  · Push decision to client via API gateway response (sync, <3 ms)                      │    │    │
│  │   │  · Trigger async: audit log, SIEM, analyst queue, model feedback loop                   │    │    │
│  │   └─────────────────────────────────────────────────────────────────────────────────────────┘    │    │
│  │                                                                                                  │    │    │
│  └──────────────────────────────────────────────────────────────────────────────────────────────────┘    │
│                                                                                                       │
│  ┌──────────────────────────────────────────────────────────────────────────────────────────────────┐  │
│  │  ASYNC / BATCH LAYER  (not on critical path)                                                     │  │
│  │                                                                                                  │  │
│  │  ┌────────────┐  ┌────────────────┐  ┌───────────────┐  ┌──────────────┐  ┌───────────────┐    │  │
│  │  │ Flink /    │  │  Nightly ML    │  │  Alerting &    │  │  Audit /     │  │  Model        │    │  │
│  │  │ Spark      │  │  retraining    │  │  PagerDuty     │  │  Compliance  │  │  Drift        │    │  │
│  │  │  ETL       │  │  (Airflow /    │  │  integration   │  │  store       │  │  Monitor      │    │  │
│  │  │  → Data    │  │  Dagster)      │  │                │  │  (Cassandra) │  │  (Evidently / │    │  │
│  │  │  Warehouse │  │  → new model   │  │                │  │             │  │  whylogs)      │    │  │
│  │  │  (Snowflake│  │  canary → prod │  │                │  │             │  │                │    │  │
│  │  │  / Iceberg)│  │                │  │                │  │             │  │                │    │  │
│  │  └────────────┘  └────────────────┘  └───────────────┘  └──────────────┘  └───────────────┘    │  │
│  └──────────────────────────────────────────────────────────────────────────────────────────────────┘  │
│                                                                                                       │
└───────────────────────────────────────────────────────────────────────────────────────────────────────┘
```

### Critical-Path Latency Budget (sub-100 ms, P99)

```
  0 ms   10    20    30    45    50    75    80    90    100 ms
  │      │     │     │     │     │     │     │     │      │
  ▼      ▼     ▼     ▼     ▼     ▼     ▼     ▼     ▼      ▼
  Edge  GW   Ingest  Enrich Rule  ML   Decision Resp  Buffer
  geo-  auth  Kafka  feats  eval  score  +write  back
  route  +dedup +emit  <5ms  <5ms <25ms  <5ms  <3ms
  ~10ms ~5ms  ~5ms         (only
                          if rules
                          pass)

  Total worst case ≈ 83 ms   ·   Typical P50 ≈ 35 ms   ·   P99 < 95 ms
```

---

## 2. Technology Stack

| Layer | Primary | Rationale / Fallback |
|---|---|---|
| **Edge / DNS** | CloudFlare + AWS Route53 latency-based | Anycast, <2 ms to 95 % of users, DDoS mitigation free |
| **API Gateway** | **Envoy** (proxy) + custom Go auth filter | Lower overhead than Kong at 100 K rps; per-tenant rate limits in Lua/Go filter |
| **Message Bus** | **Apache Pulsar** (primary) / Kafka fallback | Multi-tenant topics, geo-replication built-in, tiered storage; Kafka if team already runs it |
| **Stream Processing** | **Apache Flink 1.18+** (stateful, at-least-once) | Exactly-once semantics with RocksDB state backend; 100 K TPS per job with 64-task slots |
| **Hot Feature Store** | **Redis Cluster** (8-node, 64 GB/node, per region) | P99 < 1 ms for `GET`/`INCR`; stores velocity counters, device fingerprints, IP reputation |
| **Cold / Historical Store** | **ScyllaDB** (or Cassandra 4.1) | 90-day transaction history, per-customer feature vectors; partition by `customer_id` |
| **Graph (peer networks)** | **Neo4j 5.x** (or Kùzu for embedded) | Money-mule / ring detection; async lookup, 10 ms budget |
| **Rule Engine** | **Drools 8** (Java) or custom **Go ruleset** compiled at build time | Hot-reload via config server; no JVM GC pauses in Go path |
| **ML Inference** | **ONNX Runtime** (CPU, AVX-512) / **TensorRT** (GPU optional) | LightGBM + 2-layer MLP; 200 features; P99 < 15 ms on 8-vCPU; 2 active + 1 warm per region |
| **Model Registry / Serving** | **MLflow** + **KServe** or custom Go server | Canary slots, A/B split, one-click rollback |
| **Orchestration** | **Kubernetes** (EKS / GKE / AKS) with **Karpenter** autoscaler | GPU nodes for canary models; CPU nodes for rules; HPA on Kafka lag + p99 |
| **Config / Feature Flags** | **HashiCorp Vault** + **Unleash** | Secrets, model version toggles, per-tenant rule sets |
| **Observability** | **OpenTelemetry** → **Grafana Tempo** (traces) + **Mimir** (metrics) + **Loki** (logs) | Per-txn trace IDs; p99 dashboards per stage; SLO burn-rate alerts |
| **Async ETL / DW** | **Flink CDC** → **Apache Iceberg** on S3/GCS → **Snowflake / Trino** | Batch analytics, model retraining data; not on critical path |
| **CI/CD** | **GitHub Actions** + **ArgoCD** (GitOps) | Canary deploys, model promotion pipeline |
| **IaC** | **Terraform** + **Crossplane** | Multi-region, multi-cloud; reproducible |

---

## 3. Scaling Strategy

### 3.1 Horizontal Partitioning

```
┌──────────────────────────────────────────────────────────────────────┐
│  PARTITION KEY: customer_id  (or txn_id for raw ingestion)           │
│                                                                      │
│  128 partitions per Kafka/Pulsar topic per region                   │
│  100 000 TPS / 128 ≈ 781 TPS per partition  ← trivial for any      │
│  consumer group                                                      │
│                                                                      │
│  Flink job: 64 task slots × 3 regions = 192 parallel sub-tasks      │
│  Each sub-task handles ~520 TPS  →  headroom 5×                     │
└──────────────────────────────────────────────────────────────────────┘
```

### 3.2 Autostacking

| Trigger | Action |
|---|---|
| Kafka/Pulsar consumer lag > 5 s on any partition | Karpenter adds 2 Flink task-manager pods (same AZ, different AZ) |
| P99 latency on any stage > 70 ms (sliding 60 s window) | HPA scales that stage's pod count by 2 × current |
| Redis cluster memory > 75 % | Auto-shard: add 2 nodes, rehash |
| GPU node utilisation > 80 % (model inference) | Add 1 GPU pod; shift 10 % traffic to warm CPU replica |
| Per-tenant spike (one customer 10× volume) | Per-tenant rate limiter in Envoy sheds excess; alert SRE |

### 3.3 Geographic / Multi-Region

- **3 active regions** (US-E, EU-W, AP-SE), each fully independent.
- **Pulsar geo-replication** keeps `raw_txns` replicated across regions within 2 s.
- Feature stores are **region-local** (no cross-region reads on hot path); nightly sync of cold features via Flink CDC.
- DNS latency-based routing sends each user to nearest region; failover to peer in < 30 s via Route53 health checks.
- **Data-sovereignty**: EU traffic never leaves EU for PII fields; a redaction layer strips PII before geo-replication.

### 3.4 Capacity Planning (per region, 100 K TPS)

```
  Envoy proxies:          6 × 4-vCPU (2 active, 2 spare, 2 warming)
  Pulsar brokers:         3 × 8-vCPU, 512 GB SSD  (3-replica, quorum)
  Flink job-manager:      2 × 4-vCPU (leader/standby)
  Flink task-managers:   16 × 8-vCPU, 32 GB RAM (RocksDB state ≤ 4 GB/pod)
  Redis cluster:          8 × 16-vCPU, 64 GB RAM (3 masters, 5 replicas)
  ScyllaDB:              6 × 12-vCPU, 1 TB SSD
  ML inference pods:      4 × 8-vCPU (ONNX CPU) + 2 × T4 GPU (canary/sequence model)
  Graph (Neo4j):         3 × 8-vCPU, 64 GB (1 primary, 2 replicas)
  ─────────────────────────────────────────
  Total per region:      ~55 pods, ~480 vCPU, ~1.2 TB RAM  →  ~$18 K/mo (cloud)
```

---

## 4. Key Challenges & Practical Solutions

### 4.1 Sub-100 ms at P99, Not P50

| Problem | Solution |
|---|---|
| GC pauses (Java Flink, Drools) | Use Go for rule engine; set Flink to G1GC with 4 GB heap, `-XX:MaxGCPauseMillis=10`; pin RocksDB to separate cgroup; run ONNX in separate process via gRPC (no JVM on inference path) |
| Redis hot-key (one merchant with 100 K txns/min) | Shard Redis by `hash(merchant_id + slot)`; use Redis Cluster with hash tags; add local LRU cache (Caffeine, 1 s TTL) in Flink operators |
| Network jitter between pods | Run all critical-path services on the **same VPC, same AZ** via pod anti-affinity `topologySpreadConstraints`; use `eBPF`-based CNI (Cilium) to avoid `iptables` lookup cost |
| Tail latency from graph service | Graph lookup is **async side-input** with 10 ms timeout → returns empty on timeout; never blocks the main path |

### 4.2 99.99 % Availability (3.65 min downtime / year)

| Problem | Solution |
|---|---|
| Single region outage | 3 active regions; DNS failover in < 30 s; each region is fully independent (no cross-region dependency on hot path) |
| Kafka/Pulsar broker failure | 3-replica quorum; ISR/min-ISR = 2; Flink consumer group auto-rebalances in < 2 s |
| Flink job crash / checkpoint failure | RocksDB incremental checkpoints to S3 every 30 s; job auto-restarts from last checkpoint (< 10 s recovery); `restart-strategy: fixed-delay, 3` |
| Model serving failure | 2 active + 1 warm replica per region; circuit-breaker in Flink → fall back to rule-only decision; alert + auto-rollback to last known-good model version |
| Redis cluster split | Use `min-replicas 2`, `min-replicas-available 1`; Flink operator catches `NOGROUP` error, retries with local cache for 5 s |
| Config / rule update pushes bad rule | All rule changes go through **shadow mode** (score only, no action) for 1 h; canary → 10 % → 100 %; one-command rollback via Config Server |

### 4.3 Consistency vs. Latency

| Problem | Solution |
|---|---|
| Feature staleness (velocity counter off by 1 s) | Accept eventual consistency on the hot path; velocity window is 5 min → 1 s skew is < 0.3 % error; correct in async ETL for batch analytics |
| Double-counting on retries | Idempotency key = `txn_id` in Pulsar; Flink dedup operator with RocksDB state (5 min TTL); gateway-level request dedup via `Idempotency-Key` header |
| Cross-region data divergence | Region-local features are authoritative for that region's decisions; nightly reconciliation job flags discrepancies; no cross-region writes on hot path |

### 4.4 Security & Compliance

| Problem | Solution |
|---|---|
| PII in logs / traces | OpenTelemetry PII redaction middleware; transaction amount masked in traces; full PII only in audit store (encrypted, access-controlled) |
| PCI-DSS / GDPR | Encryption at rest (KMS / AWS KMS) + in transit (mTLS everywhere); EU data residency via region-lock; data-retention policies enforced in ScyllaDB TTL |
| Model / rule IP protection | Models stored in encrypted artifact store; inference pods have no filesystem access to weights after load; rules in encrypted config |

### 4.5 Model Drift & Adversarial Evasion

| Problem | Solution |
|---|---|
| Conventional fraud drops, new pattern emerges | **Evidently / whylogs** monitors feature distributions per 5-min window; SLO on KS-test p-value; auto-alert to ML team |
| Adversary changes transaction patterns | Weekly automated retraining (Airflow/Dagster) from last 30 days labelled data; A/B test new model on 5 % traffic; promote if F1 > current |
| Label lag (fraud confirmed 30 days later) | Use **semi-supervised** approach: anomaly detection (Isolation Forest, autoencoder) for unsupervised signal; supervised GBDT for known patterns; blend scores |

### 4.6 Testing & Confidence

| Technique | How |
|---|---|
| **Chaos engineering** | Monthly: kill a Flink task-manager, a Redis master, a Pulsar broker; validate SLO holds |
| **Load testing** | k6 / Gatling simulating 150 K TPS for 30 min (1.5× headroom) per region; measure P99 |
| **Canary deployments** | 1 % traffic → new model/rule version; compare approval rate, latency; auto-rollback on regression |
| **Post-mortem** | Every SLO burn > 1 % triggers blameless post-mortem; track MTTR |

---

## 5. Operational Runbook (Condensed)

```
SLO:  P99 < 100 ms  ·  Availability ≥ 99.99 %  ·  False-negative rate < 0.5 %

Alerts (PagerDuty):
  • Consumer lag > 5 s on any partition          → SEV-1
  • P99 latency > 80 ms (60 s window)           → SEV-1
  • Model error rate > 1 % or OOM               → SEV-1, auto-rollback
  • Redis failover triggered                    → SEV-2
  • Rule config change fails shadow validation   → SEV-2, block promotion
  • Flink checkpoint > 60 s                     → SEV-2

Runbook per alert lives in Ops Wiki, linked from alert metadata.
On-call rotates every 48 h, 2-person bridge for SEV-1.
```

---

This design keeps **everything on the 100 ms critical path in-memory** (Redis + Flink state + ONNX), pushes all heavy I/O (Cassandra, graph, warehouse) to the **async path**, and uses **three independent active regions** so no single failure can breach the 99.99 % target. The total engineering team to build and operate this at scale is roughly **8–12 people** (3 infra/platform, 3 ML/data, 2 SRE, 2 product/backend), which is realistic for a team that already runs Kafka and Kubernetes.

## 4. Performance Monitoring and Resource Usage

Let's monitor the performance characteristics of different models:

In [4]:
import time
import psutil
import matplotlib.pyplot as plt

def benchmark_model(model_name, prompt, chat_function):
    """Benchmark model performance"""
    # Get initial memory usage
    initial_memory = psutil.virtual_memory().used / (1024**3)  # GB
    
    # Time the response
    start_time = time.time()
    response = chat_function(prompt)
    end_time = time.time()
    
    # Get final memory usage
    final_memory = psutil.virtual_memory().used / (1024**3)  # GB
    
    # Calculate metrics
    response_time = end_time - start_time
    memory_used = final_memory - initial_memory
    tokens_estimated = len(response.split())  # Rough estimate
    tokens_per_second = tokens_estimated / response_time if response_time > 0 else 0
    
    return {
        'model': model_name,
        'response_time': response_time,
        'memory_used': memory_used,
        'tokens_estimated': tokens_estimated,
        'tokens_per_second': tokens_per_second,
        'response_length': len(response)
    }

# Simple benchmark prompt
benchmark_prompt = "Explain the concept of machine learning in exactly 100 words."

# Benchmark the two course models
results = []

results.append(benchmark_model("Gemma 4 (gemma4)", benchmark_prompt, chat_with_gemma4))
results.append(benchmark_model("Qwen 3.8 27B", benchmark_prompt, chat_with_qwen))

# Display results
if results:
    for result in results:
        print(f"\n{result['model']} Performance:")
        print(f"  Response Time: {result['response_time']:.2f}s")
        print(f"  Estimated Tokens/sec: {result['tokens_per_second']:.1f}")
        print(f"  Response Length: {result['response_length']} chars")
else:
    print("No benchmark results collected")


Gemma 4 (gemma4) Performance:
  Response Time: 12.21s
  Estimated Tokens/sec: 9.5
  Response Length: 853 chars

Qwen 3.8 27B Performance:
  Response Time: 108.30s
  Estimated Tokens/sec: 0.9
  Response Length: 726 chars


## 5. Advanced Use Cases

### Tool Calling and Function Integration

In [ ]:
# Example of using models for tool calling/function generation
tool_calling_prompt = """
Create a Python class that acts as a "Smart Assistant" with the following capabilities:

1. Weather lookup (mock API call)
2. Calendar management (add/remove events)
3. Email composition
4. Web search (mock implementation)
5. File operations (create, read, update files)

The class should:
- Have a unified interface for natural language commands
- Parse user intent and route to appropriate functions
- Handle errors gracefully
- Return structured responses

Include example usage showing how to:
- "What's the weather in New York?"
- "Schedule a meeting for tomorrow at 2 PM"
- "Write an email to John about the project update"
"""

response = chat_with_qwen(tool_calling_prompt)
display(Markdown(response))

### RAG (Retrieval-Augmented Generation) Integration

In [ ]:
rag_prompt = """
Design and implement a RAG (Retrieval-Augmented Generation) system that:

1. Ingests documents from multiple formats (PDF, Word, HTML, Markdown)
2. Creates embeddings using sentence-transformers
3. Stores embeddings in a vector database (Chroma or FAISS)
4. Implements semantic search with relevance scoring
5. Combines retrieved context with LLM generation

Requirements:
- Handle documents up to 10MB
- Support real-time updates
- Implement caching for performance
- Include source attribution

Provide complete Python implementation with:
- Document processing pipeline
- Vector storage and retrieval
- Integration with local LLM
- Example usage
"""

response = chat_with_gemma4(rag_prompt)
display(Markdown(response))

For which model and size to pick on your hardware, see the [2026 local model guide](../best-local-models-2026.md).

## Conclusion

This notebook demonstrated the practical capabilities of the two model families this course uses:

- **Gemma 4**: The course default, from `gemma4:e4b` on CPU to `gemma4:31b` on a 24 GB GPU
- **Qwen 3.8 27B**: The tool-calling and coding alternative

Each model has unique strengths, and the choice depends on your specific use case, available hardware, and performance requirements.

For more detailed information, see the accompanying [`best-local-models-2026.md`](../best-local-models-2026.md) guide.